# 21A PatchCore EfficientNet-B0 All-in-One

This notebook is designed for Modal on an A10 GPU and is fully self-contained.

What it does in one run:

- reads raw `LSWMD.pkl`
- rebuilds the shared `40k / 5k / 5k + 250` split used by the report notebooks
- builds the EfficientNet-B0 PatchCore memory bank
- scores validation and test sets with the same raw-score threshold rule used in notebook `21`
- saves CSV, JSON, and checkpoint artifacts under `/output`

A10 runtime note:

- the experiment itself stays aligned with notebook `21`: same split, same EfficientNet PatchCore setup, same threshold rule
- the runtime defaults are pushed higher for A10 inference throughput: larger batch size, larger score chunk, CUDA+AMP enabled, and worker prefetching
- if you still hit OOM on your Modal image, lower `batch_size` first, then `score_chunk`


In [2]:
from __future__ import annotations

import copy
import json
import pickle
import random
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

LABEL_NORMAL = "none"
LABEL_DEFECT = "pattern"

CONFIG: dict[str, Any] = {
    "run": {
        "variant_name": "effnet_b0_patchcore",
        "raw_pickle": "LSWMD.pkl",
        "output_dir": "/output/patchcore_efficientnet_b0_5pct_all_in_one",
        "seed": 42,
        "rebuild_dataset": False,
    },
    "split": {
        "image_size": 64,
        "train_normal_count": 40_000,
        "val_normal_count": 5_000,
        "test_normal_count": 5_000,
        "test_defect_fraction_of_test_normals": 0.05,
    },
    "model": {
        "device": "cuda",
        "model_input_size": 224,
        "batch_size": 256,
        "num_workers": 4,
        "persistent_workers": True,
        "prefetch_factor": 2,
        "memory_bank_max_patches": 240_000,
        "score_chunk": 2048,
        "patchcore_nn_k": 3,
        "topk_patch_ratio": 0.02,
        "effnet_mid_feature_idx": 3,
        "effnet_deep_feature_idx": 6,
        "patch_embed_dim": 512,
        "amp": True,
    },
    "scoring": {
        "threshold_quantile": 0.95,
    },
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)


def read_legacy_pickle(path: str | Path) -> pd.DataFrame:
    sys.modules["pandas.indexes"] = core_indexes
    with Path(path).open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def unwrap_legacy_value(value: Any) -> str:
    if value is None:
        return ""
    if hasattr(value, "size") and getattr(value, "size") == 0:
        return ""
    if hasattr(value, "tolist"):
        value = value.tolist()
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return str(value).strip()


def normalize_map(wafer_map: np.ndarray, image_size: int) -> np.ndarray:
    wafer_map = np.asarray(wafer_map, dtype=np.float32) / 2.0
    tensor = torch.from_numpy(wafer_map).unsqueeze(0).unsqueeze(0)
    return F.interpolate(tensor, size=(image_size, image_size), mode="nearest").squeeze(0).squeeze(0).numpy()


def infer_label_from_row(row: pd.Series) -> str | None:
    failure = unwrap_legacy_value(row.get("failureType", "")).lower()
    if failure == "none":
        return LABEL_NORMAL
    if failure:
        return LABEL_DEFECT
    return None


def build_split_from_raw(config: dict[str, Any]) -> dict[str, np.ndarray]:
    raw_path = Path(config["run"]["raw_pickle"])
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw pickle not found: {raw_path}")

    image_size = int(config["split"]["image_size"])
    train_n = int(config["split"]["train_normal_count"])
    val_n = int(config["split"]["val_normal_count"])
    test_n = int(config["split"]["test_normal_count"])
    defect_fraction = float(config["split"]["test_defect_fraction_of_test_normals"])
    requested_defects = max(1, int(round(test_n * defect_fraction)))
    seed = int(config["run"]["seed"])

    df = read_legacy_pickle(raw_path).copy()
    df["failureTypeText"] = df["failureType"].map(unwrap_legacy_value)
    df["label"] = df.apply(infer_label_from_row, axis=1)
    df = df[df["label"].notna()].reset_index(drop=True)

    normal_df = df[df["label"] == LABEL_NORMAL].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    defect_df = df[df["label"] == LABEL_DEFECT].sample(frac=1.0, random_state=seed).reset_index(drop=True)

    required_normals = train_n + val_n + test_n
    if len(normal_df) < required_normals:
        raise ValueError(f"Need {required_normals} normals but found {len(normal_df)}")
    if len(defect_df) < requested_defects:
        raise ValueError(f"Need {requested_defects} defects but found {len(defect_df)}")

    train_df = normal_df.iloc[:train_n].copy()
    val_df = normal_df.iloc[train_n : train_n + val_n].copy()
    test_normal_df = normal_df.iloc[train_n + val_n : train_n + val_n + test_n].copy()
    test_defect_df = defect_df.iloc[:requested_defects].copy()
    test_df = pd.concat([test_normal_df, test_defect_df], ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    print(
        "Prepared split | "
        f"train normals={len(train_df)} | "
        f"val normals={len(val_df)} | "
        f"test normals={len(test_normal_df)} | "
        f"test defects={len(test_defect_df)}"
    )

    def to_arrays(frame: pd.DataFrame, split_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        arrays = np.stack(
            [
                normalize_map(np.asarray(row["waferMap"]), image_size=image_size)
                for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f"prepare_{split_name}", leave=True)
            ]
        )
        labels = (frame["label"].values == LABEL_DEFECT).astype(np.int64)
        defect_types = frame["failureTypeText"].fillna("unlabeled").replace("", "unlabeled").to_numpy()
        return arrays, labels, defect_types

    train_x, train_y, train_defect_types = to_arrays(train_df, "train")
    val_x, val_y, val_defect_types = to_arrays(val_df, "val")
    test_x, test_y, test_defect_types = to_arrays(test_df, "test")

    return {
        "train_x": train_x,
        "train_y": train_y,
        "train_defect_types": train_defect_types,
        "val_x": val_x,
        "val_y": val_y,
        "val_defect_types": val_defect_types,
        "test_x": test_x,
        "test_y": test_y,
        "test_defect_types": test_defect_types,
    }


class EfficientNetArrayDataset(Dataset):
    def __init__(self, arrays: np.ndarray, labels: np.ndarray, model_input_size: int) -> None:
        self.arrays = arrays.astype(np.float32)
        self.labels = labels.astype(np.int64)
        self.model_input_size = int(model_input_size)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.arrays[index])
        states = torch.clamp(torch.round(x * 2.0), 0, 2).to(dtype=torch.long)
        one_hot = F.one_hot(states, num_classes=3).permute(2, 0, 1).to(dtype=torch.float32)
        if one_hot.shape[-1] != self.model_input_size or one_hot.shape[-2] != self.model_input_size:
            one_hot = F.interpolate(
                one_hot.unsqueeze(0),
                size=(self.model_input_size, self.model_input_size),
                mode="nearest",
            ).squeeze(0)
        y = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return one_hot, y


class EfficientNetPatchCoreModel(nn.Module):
    def __init__(
        self,
        *,
        model_input_size: int,
        mid_feature_idx: int,
        deep_feature_idx: int,
        patch_embed_dim: int,
        topk_ratio: float,
        nn_k: int,
        query_chunk_size: int,
        amp_enabled: bool,
    ) -> None:
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        self.mid_feature_idx = int(mid_feature_idx)
        self.deep_feature_idx = int(deep_feature_idx)
        self.patch_embed_dim = int(patch_embed_dim)
        self.topk_ratio = float(topk_ratio)
        self.nn_k = int(nn_k)
        self.query_chunk_size = int(query_chunk_size)
        self.amp_enabled = bool(amp_enabled)

        with torch.inference_mode():
            dummy = torch.zeros(1, 3, model_input_size, model_input_size)
            x = dummy
            f_mid = None
            f_deep = None
            for i, block in enumerate(self.features):
                x = block(x)
                if i == self.mid_feature_idx:
                    f_mid = x
                if i == self.deep_feature_idx:
                    f_deep = x

        if f_mid is None or f_deep is None:
            raise ValueError(
                f"Invalid EfficientNet feature indices: mid={self.mid_feature_idx}, deep={self.deep_feature_idx}"
            )

        in_dim = int(f_mid.shape[1] + f_deep.shape[1])
        self.patch_grid = tuple(int(dim) for dim in f_mid.shape[-2:])
        self.feature_dim = self.patch_embed_dim
        self.proj = nn.Linear(in_dim, self.patch_embed_dim, bias=False)
        self.register_buffer("memory_bank", torch.empty(0, self.feature_dim))

        for parameter in self.features.parameters():
            parameter.requires_grad_(False)
        for parameter in self.proj.parameters():
            parameter.requires_grad_(False)

    @property
    def patches_per_image(self) -> int:
        return int(self.patch_grid[0] * self.patch_grid[1])

    def forward_feature_maps(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        f_mid = None
        f_deep = None
        for i, block in enumerate(self.features):
            x = block(x)
            if i == self.mid_feature_idx:
                f_mid = x
            if i == self.deep_feature_idx:
                f_deep = x
        if f_mid is None or f_deep is None:
            raise RuntimeError("Failed to collect EfficientNet feature maps.")
        return f_mid, f_deep

    def patch_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        amp_enabled = self.amp_enabled and x.device.type == "cuda"
        with torch.inference_mode():
            with torch.autocast(device_type=x.device.type, dtype=torch.float16, enabled=amp_enabled):
                f_mid, f_deep = self.forward_feature_maps(x)
                f_deep_up = F.interpolate(
                    f_deep,
                    size=f_mid.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )
                emb = torch.cat([f_mid, f_deep_up], dim=1)
                emb = emb.permute(0, 2, 3, 1).reshape(-1, emb.shape[1])
                emb = self.proj(emb)
            emb = F.normalize(emb.float(), p=2, dim=1)
        return emb.reshape(x.shape[0], self.patches_per_image, self.feature_dim)

    def set_memory_bank(self, memory_bank: torch.Tensor) -> None:
        if memory_bank.ndim != 2 or memory_bank.shape[1] != self.feature_dim:
            raise ValueError(
                f"memory_bank must have shape (N, {self.feature_dim}), got {tuple(memory_bank.shape)}"
            )
        normalized = F.normalize(memory_bank.to(dtype=torch.float32), p=2, dim=1)
        self.memory_bank = normalized.to(device=self.memory_bank.device, dtype=self.memory_bank.dtype)

    def nearest_patch_distances(self, patch_embeddings: torch.Tensor) -> torch.Tensor:
        if self.memory_bank.numel() == 0:
            raise ValueError("memory_bank is empty.")

        batch_size, patch_count, _ = patch_embeddings.shape
        flat_queries = patch_embeddings.reshape(-1, self.feature_dim)
        bank_t = self.memory_bank.t().contiguous()
        outputs = []

        for start in range(0, flat_queries.shape[0], self.query_chunk_size):
            query_chunk = flat_queries[start : start + self.query_chunk_size]
            sim = query_chunk @ bank_t
            k = min(self.nn_k, sim.shape[1])
            best_sim = sim.topk(k=k, dim=1).values
            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best_sim, min=0.0))
            outputs.append(dist.mean(dim=1))

        return torch.cat(outputs, dim=0).reshape(batch_size, patch_count)

    def reduce_patch_distances(self, patch_distances: torch.Tensor) -> torch.Tensor:
        topk = max(1, int(round(patch_distances.shape[1] * self.topk_ratio)))
        topk = min(topk, patch_distances.shape[1])
        return torch.topk(patch_distances, k=topk, dim=1).values.mean(dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patch_embeddings = self.patch_embeddings(x)
        patch_distances = self.nearest_patch_distances(patch_embeddings)
        return self.reduce_patch_distances(patch_distances)


def prepare_output_dir(config: dict[str, Any]) -> Path:
    output_dir = Path(config["run"]["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def dataset_cache_path(config: dict[str, Any]) -> Path:
    return prepare_output_dir(config) / "dataset_cache.npz"


def save_dataset_cache(dataset: dict[str, np.ndarray], cache_path: Path) -> None:
    np.savez_compressed(cache_path, **dataset)
    print(f"Saved dataset cache to {cache_path}")


def load_dataset_cache(cache_path: Path) -> dict[str, np.ndarray]:
    cached = np.load(cache_path, allow_pickle=True)
    dataset = {key: cached[key] for key in cached.files}
    print(f"Loaded dataset cache from {cache_path}")
    return dataset


def prepare_dataset(config: dict[str, Any], rebuild: bool = False) -> dict[str, np.ndarray]:
    set_seed(int(config["run"]["seed"]))
    cache_path = dataset_cache_path(config)
    if cache_path.exists() and not rebuild:
        return load_dataset_cache(cache_path)
    dataset = build_split_from_raw(config)
    save_dataset_cache(dataset, cache_path)
    return dataset


def make_loaders(dataset: dict[str, np.ndarray], config: dict[str, Any], device: torch.device) -> dict[str, DataLoader]:
    bs = int(config["model"]["batch_size"])
    nw = int(config["model"]["num_workers"])
    pin = device.type == "cuda"
    model_input_size = int(config["model"]["model_input_size"])
    loader_kwargs: dict[str, Any] = {
        "batch_size": bs,
        "num_workers": nw,
        "pin_memory": pin,
    }
    if nw > 0:
        loader_kwargs["persistent_workers"] = bool(config["model"].get("persistent_workers", True))
        loader_kwargs["prefetch_factor"] = int(config["model"].get("prefetch_factor", 2))
    return {
        "train": DataLoader(EfficientNetArrayDataset(dataset["train_x"], dataset["train_y"], model_input_size), shuffle=False, **loader_kwargs),
        "val": DataLoader(EfficientNetArrayDataset(dataset["val_x"], dataset["val_y"], model_input_size), shuffle=False, **loader_kwargs),
        "test": DataLoader(EfficientNetArrayDataset(dataset["test_x"], dataset["test_y"], model_input_size), shuffle=False, **loader_kwargs),
    }


def collect_memory_bank(
    model: EfficientNetPatchCoreModel,
    loader: DataLoader,
    device: torch.device,
    target_size: int,
    seed: int,
) -> torch.Tensor:
    if target_size <= 0:
        raise ValueError("target_size must be positive")

    patch_batches: list[torch.Tensor] = []
    total_seen_patches = 0
    estimated_total_patches = model.patches_per_image * len(loader.dataset)
    sample_ratio = min(1.0, target_size / estimated_total_patches)
    generator = torch.Generator().manual_seed(seed)

    print(
        {
            "estimated_total_patches": int(estimated_total_patches),
            "sample_ratio": round(sample_ratio, 6),
        }
    )

    model.eval()
    progress = tqdm(loader, desc="memory_bank", total=len(loader), leave=True)
    with torch.inference_mode():
        for inputs, labels in progress:
            inputs = inputs.to(device, non_blocking=device.type == "cuda")
            labels = labels.to(device)
            normal_mask = labels == 0
            if not torch.any(normal_mask):
                continue

            embeddings = model.patch_embeddings(inputs[normal_mask]).reshape(-1, model.feature_dim)
            total_seen_patches += int(embeddings.shape[0])
            embeddings = embeddings.cpu()

            if sample_ratio < 1.0:
                keep_n = max(1, int(round(embeddings.shape[0] * sample_ratio)))
                keep_idx = torch.randperm(embeddings.shape[0], generator=generator)[:keep_n]
                embeddings = embeddings[keep_idx]

            patch_batches.append(embeddings)
            progress.set_postfix(seen_patches=total_seen_patches, kept_batches=len(patch_batches))

    if not patch_batches:
        raise ValueError("Could not build memory bank because no normal embeddings were collected.")

    memory_bank = torch.cat(patch_batches, dim=0)
    print({"observed_raw_patches": int(total_seen_patches), "sampled_before_trim": int(memory_bank.shape[0])})

    if memory_bank.shape[0] > target_size:
        keep_gen = torch.Generator().manual_seed(seed)
        keep = torch.randperm(memory_bank.shape[0], generator=keep_gen)[:target_size]
        memory_bank = memory_bank[keep]

    return memory_bank


def collect_scores(
    model: EfficientNetPatchCoreModel,
    loader: DataLoader,
    device: torch.device,
    desc: str,
) -> pd.DataFrame:
    rows = []
    model.eval()
    progress = tqdm(loader, desc=desc, total=len(loader), leave=True)
    with torch.inference_mode():
        for inputs, labels in progress:
            inputs = inputs.to(device, non_blocking=device.type == "cuda")
            scores = model(inputs).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                rows.append({"score": float(score), "is_anomaly": int(label)})
            progress.set_postfix(rows=len(rows))
    return pd.DataFrame(rows)


def summarize_threshold_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    scores = np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    predicted = (scores > threshold).astype(int)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "predicted_anomalies": int(predicted.sum()),
        "confusion_matrix": [[tn, fp], [fn, tp]],
    }


def sweep_threshold_metrics(labels: np.ndarray, scores: np.ndarray) -> tuple[pd.DataFrame, dict[str, Any]]:
    scores = np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    thresholds = np.unique(scores)
    rows = []
    for threshold in thresholds:
        metrics = summarize_threshold_metrics(labels, scores, float(threshold))
        rows.append(
            {
                "threshold": float(threshold),
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "predicted_anomalies": metrics["predicted_anomalies"],
            }
        )
    sweep_df = pd.DataFrame(rows).sort_values(["f1", "recall", "precision", "threshold"], ascending=[False, False, False, True]).reset_index(drop=True)
    best_row = sweep_df.iloc[0].to_dict()
    return sweep_df, best_row


def build_defect_breakdown(defect_types: np.ndarray, labels: np.ndarray, scores: np.ndarray, threshold: float) -> pd.DataFrame:
    eval_df = pd.DataFrame(
        {
            "defect_type": defect_types,
            "is_anomaly": labels.astype(int),
            "score": scores,
        }
    )
    eval_df["predicted_anomaly"] = (eval_df["score"] > threshold).astype(int)
    defect_only = eval_df[eval_df["is_anomaly"] == 1].copy()
    grouped = defect_only.groupby("defect_type").agg(
        count=("defect_type", "size"),
        detected=("predicted_anomaly", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
    )
    grouped["missed"] = grouped["count"] - grouped["detected"]
    grouped["recall"] = grouped["detected"] / grouped["count"].clip(lower=1)
    return grouped.sort_values(["recall", "count", "mean_score"], ascending=[True, False, True]).reset_index()


def run_variant(dataset: dict[str, np.ndarray], config: dict[str, Any]) -> dict[str, Any]:
    set_seed(int(config["run"]["seed"]))
    device = resolve_device(str(config["model"]["device"]))
    loaders = make_loaders(dataset, config, device)
    model = EfficientNetPatchCoreModel(
        model_input_size=int(config["model"]["model_input_size"]),
        mid_feature_idx=int(config["model"]["effnet_mid_feature_idx"]),
        deep_feature_idx=int(config["model"]["effnet_deep_feature_idx"]),
        patch_embed_dim=int(config["model"]["patch_embed_dim"]),
        topk_ratio=float(config["model"]["topk_patch_ratio"]),
        nn_k=int(config["model"]["patchcore_nn_k"]),
        query_chunk_size=int(config["model"]["score_chunk"]),
        amp_enabled=bool(config["model"]["amp"]),
    ).to(device).eval()

    build_start = time.perf_counter()
    memory_bank = collect_memory_bank(
        model=model,
        loader=loaders["train"],
        device=device,
        target_size=int(config["model"]["memory_bank_max_patches"]),
        seed=int(config["run"]["seed"]),
    )
    model.set_memory_bank(memory_bank.to(device))
    build_seconds = time.perf_counter() - build_start

    score_val_start = time.perf_counter()
    val_scores_df = collect_scores(model, loaders["val"], device, desc="score_val")
    score_val_seconds = time.perf_counter() - score_val_start

    score_test_start = time.perf_counter()
    test_scores_df = collect_scores(model, loaders["test"], device, desc="score_test")
    score_test_seconds = time.perf_counter() - score_test_start

    val_normal_scores = val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"].to_numpy()
    threshold = float(np.quantile(val_normal_scores, float(config["scoring"]["threshold_quantile"])))
    labels = test_scores_df["is_anomaly"].to_numpy()
    scores = test_scores_df["score"].to_numpy()
    metrics = summarize_threshold_metrics(labels, scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)
    defect_breakdown_df = build_defect_breakdown(dataset["test_defect_types"], labels, scores, threshold)

    summary = {
        "variant": str(config["run"]["variant_name"]),
        "name": "PatchCore-EfficientNetB0-5pct",
        "threshold_policy": "val_normal_quantile_raw",
        "threshold_quantile": float(config["scoring"]["threshold_quantile"]),
        "threshold": float(threshold),
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "best_sweep_threshold": float(best_sweep["threshold"]),
        "best_sweep_precision": float(best_sweep["precision"]),
        "best_sweep_recall": float(best_sweep["recall"]),
        "best_sweep_f1": float(best_sweep["f1"]),
        "predicted_anomalies": int(metrics["predicted_anomalies"]),
        "memory_bank_size": int(model.memory_bank.shape[0]),
        "patches_per_image": int(model.patches_per_image),
        "feature_dim": int(model.feature_dim),
        "patch_grid_h": int(model.patch_grid[0]),
        "patch_grid_w": int(model.patch_grid[1]),
        "build_seconds": float(build_seconds),
        "score_val_seconds": float(score_val_seconds),
        "score_test_seconds": float(score_test_seconds),
        "device": str(device),
        "config": copy.deepcopy(config),
    }

    return {
        "score_df": pd.DataFrame([summary]),
        "summary": summary,
        "val_scores_df": val_scores_df,
        "test_scores_df": test_scores_df,
        "threshold_sweep_df": threshold_sweep_df,
        "defect_breakdown_df": defect_breakdown_df,
        "checkpoint": {
            "model_state_dict": model.state_dict(),
            "memory_bank": model.memory_bank.detach().cpu(),
            "summary": summary,
        },
    }


def save_variant_result(output_dir: Path, variant_name: str, result: dict[str, Any]) -> None:
    result["score_df"].to_csv(output_dir / f"{variant_name}_best_row.csv", index=False)
    result["val_scores_df"].to_csv(output_dir / f"{variant_name}_val_scores.csv", index=False)
    result["test_scores_df"].to_csv(output_dir / f"{variant_name}_test_scores.csv", index=False)
    result["threshold_sweep_df"].to_csv(output_dir / f"{variant_name}_threshold_sweep.csv", index=False)
    result["defect_breakdown_df"].to_csv(output_dir / f"{variant_name}_defect_breakdown.csv", index=False)
    with (output_dir / f"{variant_name}_summary.json").open("w", encoding="utf-8") as handle:
        json.dump(result["summary"], handle, indent=2)
    torch.save(result["checkpoint"], output_dir / f"{variant_name}_best_model.pt")
    pd.DataFrame([result["summary"]]).to_csv(output_dir / "variant_summary.csv", index=False)
    with (output_dir / "config_snapshot.json").open("w", encoding="utf-8") as handle:
        json.dump(result["summary"]["config"], handle, indent=2)
    print(f"Saved outputs for {variant_name} to {output_dir}")


def run_and_save_variant(dataset: dict[str, np.ndarray], config: dict[str, Any]) -> dict[str, Any]:
    output_dir = prepare_output_dir(config)
    variant_name = str(config["run"]["variant_name"])
    print(f"Running {variant_name}")
    result = run_variant(dataset, config)
    save_variant_result(output_dir, variant_name, result)
    return result


def load_saved_summary(config: dict[str, Any]) -> pd.DataFrame:
    summary_path = prepare_output_dir(config) / "variant_summary.csv"
    if not summary_path.exists():
        raise FileNotFoundError(f"Summary not found: {summary_path}")
    return pd.read_csv(summary_path)


pd.DataFrame([
    {
        "raw_pickle": CONFIG["run"]["raw_pickle"],
        "output_dir": CONFIG["run"]["output_dir"],
        "train_normals": CONFIG["split"]["train_normal_count"],
        "val_normals": CONFIG["split"]["val_normal_count"],
        "test_normals": CONFIG["split"]["test_normal_count"],
        "test_defect_fraction": CONFIG["split"]["test_defect_fraction_of_test_normals"],
        "model_input_size": CONFIG["model"]["model_input_size"],
        "batch_size": CONFIG["model"]["batch_size"],
        "num_workers": CONFIG["model"]["num_workers"],
        "persistent_workers": CONFIG["model"]["persistent_workers"],
        "prefetch_factor": CONFIG["model"]["prefetch_factor"],
        "memory_bank_max_patches": CONFIG["model"]["memory_bank_max_patches"],
        "score_chunk": CONFIG["model"]["score_chunk"],
        "patchcore_nn_k": CONFIG["model"]["patchcore_nn_k"],
        "topk_patch_ratio": CONFIG["model"]["topk_patch_ratio"],
        "effnet_mid_feature_idx": CONFIG["model"]["effnet_mid_feature_idx"],
        "effnet_deep_feature_idx": CONFIG["model"]["effnet_deep_feature_idx"],
        "patch_embed_dim": CONFIG["model"]["patch_embed_dim"],
        "threshold_quantile": CONFIG["scoring"]["threshold_quantile"],
    }
])


,raw_pickle,output_dir,train_normals,val_normals,test_normals,test_defect_fraction,model_input_size,batch_size,num_workers,persistent_workers,prefetch_factor,memory_bank_max_patches,score_chunk,patchcore_nn_k,topk_patch_ratio,effnet_mid_feature_idx,effnet_deep_feature_idx,patch_embed_dim,threshold_quantile
0,LSWMD.pkl,/output/patchcore_efficientnet_b0_5pct_all_in_one,40000,5000,5000,0.05,224,256,4,True,2,240000,2048,3,0.02,3,6,512,0.95


In [5]:
output_dir = prepare_output_dir(CONFIG)
cache_path = dataset_cache_path(CONFIG)
dataset = prepare_dataset(CONFIG, rebuild=bool(CONFIG["run"].get("rebuild_dataset", False)))
print(f"Output dir: {output_dir}")
print(f"Dataset cache: {cache_path}")
print({key: value.shape if hasattr(value, "shape") else type(value) for key, value in dataset.items()})


Prepared split | train normals=40000 | val normals=5000 | test normals=5000 | test defects=250


prepare_train:   0%|          | 0/40000 [00:00<?, ?it/s]

prepare_val:   0%|          | 0/5000 [00:00<?, ?it/s]

prepare_test:   0%|          | 0/5250 [00:00<?, ?it/s]

Saved dataset cache to /output/patchcore_efficientnet_b0_5pct_all_in_one/dataset_cache.npz
Output dir: /output/patchcore_efficientnet_b0_5pct_all_in_one
Dataset cache: /output/patchcore_efficientnet_b0_5pct_all_in_one/dataset_cache.npz
{'train_x': (40000, 64, 64), 'train_y': (40000,), 'train_defect_types': (40000,), 'val_x': (5000, 64, 64), 'val_y': (5000,), 'val_defect_types': (5000,), 'test_x': (5250, 64, 64), 'test_y': (5250,), 'test_defect_types': (5250,)}
Loaded dataset cache from /output/patchcore_efficientnet_b0_5pct_all_in_one/dataset_cache.npz
Output dir: /output/patchcore_efficientnet_b0_5pct_all_in_one
Dataset cache: /output/patchcore_efficientnet_b0_5pct_all_in_one/dataset_cache.npz
{'train_x': (40000, 64, 64), 'train_y': (40000,), 'train_defect_types': (40000,), 'val_x': (5000, 64, 64), 'val_y': (5000,), 'val_defect_types': (5000,), 'test_x': (5250, 64, 64), 'test_y': (5250,), 'test_defect_types': (5250,)}


In [6]:
result = run_and_save_variant(dataset, CONFIG)
display(result["score_df"])
display(result["defect_breakdown_df"].head(12))


Running effnet_b0_patchcore
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 113MB/s]


{'estimated_total_patches': 31360000, 'sample_ratio': 0.007653}


memory_bank:   0%|          | 0/157 [00:00<?, ?it/s]

{'observed_raw_patches': 31360000, 'sampled_before_trim': 240000}


score_val:   0%|          | 0/20 [00:00<?, ?it/s]

score_test:   0%|          | 0/21 [00:00<?, ?it/s]

Saved outputs for effnet_b0_patchcore to /output/patchcore_efficientnet_b0_5pct_all_in_one


,variant,name,threshold_policy,threshold_quantile,threshold,precision,recall,f1,auroc,auprc,...,memory_bank_size,patches_per_image,feature_dim,patch_grid_h,patch_grid_w,build_seconds,score_val_seconds,score_test_seconds,device,config
0,effnet_b0_patchcore,PatchCore-EfficientNetB0-5pct,val_normal_quantile_raw,0.95,0.506538,0.381313,0.604,0.467492,0.905171,0.489141,...,240000,784,512,28,28,80.712669,84.241063,84.778521,cuda,{'run': {'variant_name': 'effnet_b0_patchcore'...


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Center,50,20,0.504609,0.492717,30,0.400000
1,Edge-Loc,53,26,0.525413,0.506409,27,0.490566
2,Scratch,15,8,0.538861,0.536997,7,0.533333
3,Loc,34,19,0.531717,0.520522,15,0.558824
4,Edge-Ring,84,65,0.567183,0.550305,19,0.773810
5,Donut,7,6,0.573022,0.561511,1,0.857143
6,Random,5,5,0.629711,0.634581,0,1.000000
7,Near-full,2,2,0.614857,0.614857,0,1.000000


In [7]:
summary_df = load_saved_summary(CONFIG)
variant_name = str(CONFIG["run"]["variant_name"])
print(f"Best-row CSV: {output_dir / f'{variant_name}_best_row.csv'}")
print(f"Validation-score CSV: {output_dir / f'{variant_name}_val_scores.csv'}")
print(f"Test-score CSV: {output_dir / f'{variant_name}_test_scores.csv'}")
print(f"Threshold-sweep CSV: {output_dir / f'{variant_name}_threshold_sweep.csv'}")
print(f"Defect-breakdown CSV: {output_dir / f'{variant_name}_defect_breakdown.csv'}")
print(f"Checkpoint: {output_dir / f'{variant_name}_best_model.pt'}")
display(summary_df)


Best-row CSV: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_best_row.csv
Validation-score CSV: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_val_scores.csv
Test-score CSV: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_test_scores.csv
Threshold-sweep CSV: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_threshold_sweep.csv
Defect-breakdown CSV: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_defect_breakdown.csv
Checkpoint: /output/patchcore_efficientnet_b0_5pct_all_in_one/effnet_b0_patchcore_best_model.pt


,variant,name,threshold_policy,threshold_quantile,threshold,precision,recall,f1,auroc,auprc,...,memory_bank_size,patches_per_image,feature_dim,patch_grid_h,patch_grid_w,build_seconds,score_val_seconds,score_test_seconds,device,config
0,effnet_b0_patchcore,PatchCore-EfficientNetB0-5pct,val_normal_quantile_raw,0.95,0.506538,0.381313,0.604,0.467492,0.905171,0.489141,...,240000,784,512,28,28,80.712669,84.241063,84.778521,cuda,{'run': {'variant_name': 'effnet_b0_patchcore'...
